In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# --------------------------
# 1) Small corpus (2 docs for simplicity)
# --------------------------
docs = [
    "I love AI",
    "I love love ML"
]
labels = np.array([1, 0])   # sentiment labels: 1=positive, 0=negative

# --------------------------
# 2) Tokenize + Build vocab
# --------------------------
tokenized = [d.lower().split() for d in docs]
vocab = sorted({tok for doc in tokenized for tok in doc})
idx = {tok: i for i, tok in enumerate(vocab)}

print("Vocabulary:", vocab)
print("Index mapping:", idx)

# --------------------------
# 3) Build BoW vectors (manual dry run)
# --------------------------
X = np.zeros((len(tokenized), len(vocab)), dtype=int)

for doc_i, toks in enumerate(tokenized):
    for t in toks:
        j = idx[t]
        X[doc_i, j] += 1
    print(f"Doc {doc_i} tokens: {toks}")
    print(f"Vector: {X[doc_i].tolist()}")

print("\nFinal Document-Term Matrix (X):")
print(f"X : {X}")
print(f"Y : {labels}")
# --------------------------
# 4) Train Logistic Regression on these vectors
# --------------------------
clf = LogisticRegression(max_iter=200)
clf.fit(X, labels)
pred = clf.predict(X)

print("\nPredictions:", pred)
print("True Labels:", labels.tolist())
print("Train Accuracy:", accuracy_score(labels, pred))


Vocabulary: ['ai', 'i', 'love', 'ml']
Index mapping: {'ai': 0, 'i': 1, 'love': 2, 'ml': 3}
Doc 0 tokens: ['i', 'love', 'ai']
Vector: [1, 1, 1, 0]
Doc 1 tokens: ['i', 'love', 'love', 'ml']
Vector: [0, 1, 2, 1]

Final Document-Term Matrix (X):
X : [[1 1 1 0]
 [0 1 2 1]]
Y : [1 0]

Predictions: [1 0]
True Labels: [1, 0]
Train Accuracy: 1.0


Dry run why Doc0 → [1,1,1,0]

Step 1. Vocabulary

From all docs:
["ai", "i", "love", "ml"]

Index map:
"ai" → 0
"i" → 1
"love" → 2
"ml" → 3
So each doc vector will have 4 positions, one per vocab word.

Step 2. Doc0 tokens

Doc0 = "I love AI" → lowercase → ["i", "love", "ai"]


Step 3. Build the vector

Start with [0,0,0,0] (all counts zero).

See "i" → at index 1 → increment → [0,1,0,0]

See "love" → at index 2 → increment → [0,1,1,0]

See "ai" → at index 0 → increment → [1,1,1,0]

Final Doc0 vector
[1, 1, 1, 0]

1 at position 0 → "ai" appeared once.

1 at position 1 → "i" appeared once.

1 at position 2 → "love" appeared once.

0 at position 3 → "ml" never appeared.

That’s why each “1” means that word appeared exactly once in the document



Step 1. Vocabulary (same as before)
["ai", "i", "love", "ml"]

Index mapping:
"ai" → 0
"i" → 1
"love" → 2
"ml" → 3

Step 2. Doc1 tokens
Doc1 = "I love love ML" → lowercase → ["i", "love", "love", "ml"]

Step 3. Build the vector

Start: [0,0,0,0]

See "i" → index 1 → [0,1,0,0]

See "love" → index 2 → [0,1,1,0]

See "love" again → index 2 → [0,1,2,0]

See "ml" → index 3 → [0,1,2,1]

Final Doc1 vector
[0, 1, 2, 1]

0 at position 0 → "ai" not present.

1 at position 1 → "i" appeared once.

2 at position 2 → "love" appeared twice.

1 at position 3 → "ml" appeared once.

So the numbers in the BoW vector are just word counts per document, aligned with the vocabulary order.

# TF-IDF

In [4]:
import numpy as np
import math

# counts
doc0 = np.array([1,1,1,0], dtype=float)  # [ai, i, love, ml]
doc1 = np.array([0,1,2,1], dtype=float)
N = 2
dfs = np.array([1,2,2,1], dtype=float)   # [ai, i, love, ml]

# Classic idf = ln(N/df)
idf_classic = np.log(N / dfs)
tfidf0_classic = doc0 * idf_classic
tfidf1_classic = doc1 * idf_classic

# sklearn-style idf = ln((1+N)/(1+df)) + 1
idf_sklearn = np.log((1+N)/(1+dfs)) + 1
tfidf0_sk = doc0 * idf_sklearn
tfidf1_sk = doc1 * idf_sklearn

def l2norm(x): 
    n = np.linalg.norm(x)
    return x if n == 0 else x / n

print("Classic TF-IDF (no norm):")
print(" Doc0:", tfidf0_classic.round(4))
print(" Doc1:", tfidf1_classic.round(4))

print("\nSklearn TF-IDF (no norm):")
print(" Doc0:", tfidf0_sk.round(4))
print(" Doc1:", tfidf1_sk.round(4))

print("\nSklearn TF-IDF (L2 norm):")
print(" Doc0:", l2norm(tfidf0_sk).round(3))
print(" Doc1:", l2norm(tfidf1_sk).round(3))


Classic TF-IDF (no norm):
 Doc0: [0.6931 0.     0.     0.    ]
 Doc1: [0.     0.     0.     0.6931]

Sklearn TF-IDF (no norm):
 Doc0: [1.4055 1.     1.     0.    ]
 Doc1: [0.     1.     2.     1.4055]

Sklearn TF-IDF (L2 norm):
 Doc0: [0.705 0.502 0.502 0.   ]
 Doc1: [0.    0.379 0.757 0.532]


In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ----- Data -----
docs = ["I love AI", "I love love ML"]
y = np.array([1, 0])  # labels: Doc0=1 (pos), Doc1=0 (neg)

# ----- Fixed vocab order from our earlier dry run -----
vocab = ["ai", "i", "love", "ml"]

def bow_counts(doc, vocab):
    toks = doc.lower().split()
    return np.array([toks.count(w) for w in vocab], dtype=float)

# 1) Raw counts (TF)
X_counts = np.vstack([bow_counts(d, vocab) for d in docs])   # shape: (2, 4)

# 2) Document frequency per term
N = X_counts.shape[0]
dfs = np.count_nonzero(X_counts > 0, axis=0)  # across docs

# 3) sklearn-style IDF with smoothing: idf = ln((1+N)/(1+df)) + 1
idf = np.log((1 + N) / (1 + dfs)) + 1

# 4) TF-IDF (no sublinear TF) + L2 normalization (like TfidfVectorizer default)
X_tfidf = X_counts * idf  # elementwise multiply
norms = np.linalg.norm(X_tfidf, axis=1, keepdims=True)
X_tfidf_norm = X_tfidf / np.where(norms == 0, 1, norms)

# ----- Train Logistic Regression -----
clf = LogisticRegression(max_iter=1000)
clf.fit(X_tfidf_norm, y)

pred = clf.predict(X_tfidf_norm)
proba = clf.predict_proba(X_tfidf_norm)[:, 1]

print("Vocab order:", vocab)
print("TF-IDF (L2 norm) matrix:\n", np.round(X_tfidf_norm, 3))
print("Coefficients (per token):", dict(zip(vocab, np.round(clf.coef_[0], 3))))
print("Bias:", round(clf.intercept_[0], 3))
print("Pred:", pred.tolist())
print("Proba positive:", np.round(proba, 3).tolist())
print("Train acc:", accuracy_score(y, pred))


Vocab order: ['ai', 'i', 'love', 'ml']
TF-IDF (L2 norm) matrix:
 [[0.705 0.502 0.502 0.   ]
 [0.    0.379 0.757 0.532]]
Coefficients (per token): {'ai': 0.318, 'i': 0.056, 'love': -0.115, 'ml': -0.24}
Bias: 0.0
Pred: [1, 0]
Proba positive: [0.548, 0.452]
Train acc: 1.0


Classic TF-IDF (idf = ln(N/df))

scikit-learn style with smoothing (idf = ln((1+N)/(1+df)) + 1) and L2 normalization (default)

Data

Doc0: "I love AI" → tokens: ["i","love","ai"]

Doc1: "I love love ML" → tokens: ["i","love","love","ml"]

Vocabulary & column order: ["ai", "i", "love", "ml"]
Counts (TF as raw counts):

Doc0 → [1, 1, 1, 0]

Doc1 → [0, 1, 2, 1]

Document frequencies (df):

df("ai") = 1 (appears only in Doc0)

df("i") = 2 (appears in both)

df("love") = 2 (both)

df("ml") = 1 (only Doc1)

Total docs N = 2.


1) Classic TF-IDF (idf = ln(N/df))

idf("ai") = ln(2/1) = ln 2 ≈ 0.6931

idf("i") = ln(2/2) = ln 1 = 0

idf("love") = ln(2/2) = 0

idf("ml") = ln(2/1) = ln 2 ≈ 0.6931

TF-IDF (no normalization):

Doc0 = counts × idf =
[1*0.6931, 1*0, 1*0, 0*0.6931] = [0.6931, 0, 0, 0]

Doc1 =
[0*0.6931, 1*0, 2*0, 1*0.6931] = [0, 0, 0, 0.6931]

Intuition: words that appear in all docs (here: i, love) get idf = 0, so they vanish. On a tiny corpus this can be extreme.

(Optional) L2 normalize per document (common in practice):

Doc0 norm = √(0.6931²) = 0.6931 → normalized Doc0 ≈ [1, 0, 0, 0]

Doc1 norm = √(0.6931²) = 0.6931 → normalized Doc1 ≈ [0, 0, 0, 1]

2) scikit-learn TF-IDF (smoothing + +1 offset)

scikit-learn’s default idf (when smooth_idf=True) is:

idf(t)=ln⁡(1+N1+df(t))+1
idf(t)=ln(
1+df(t)
1+N
	​

)+1

For N = 2:

idf("ai") = ln( (1+2)/(1+1) ) + 1 = ln(3/2) + 1 ≈ 0.4055 + 1 = 1.4055

idf("i") = ln(3/3) + 1 = ln(1) + 1 = 1

idf("love") = 1 (same reason)

idf("ml") = 1.4055

TF-IDF (no sublinear TF, no norm yet):

Doc0 = [1*1.4055, 1*1, 1*1, 0] = [1.4055, 1, 1, 0]

Doc1 = [0, 1*1, 2*1, 1*1.4055] = [0, 1, 2, 1.4055]

L2 normalization (sklearn does this by default with norm='l2'):

Doc0:
norm = √(1.4055² + 1² + 1²) ≈ √(1.9755 + 1 + 1) ≈ √3.9755 ≈ 1.994
normalized ≈ [1.4055/1.994, 1/1.994, 1/1.994, 0]
≈ [0.705, 0.502, 0.502, 0.000]

Doc1:
norm = √(0² + 1² + 2² + 1.4055²) ≈ √(0 + 1 + 4 + 1.9755) ≈ √6.9755 ≈ 2.642
normalized ≈ [0/2.642, 1/2.642, 2/2.642, 1.4055/2.642]
≈ [0.000, 0.379, 0.757, 0.532]

With smoothing, even ubiquitous terms (i, love) keep idf ≈ 1 instead of 0, so they don’t vanish. L2 normalization keeps document vectors comparable in length.

# N-grams Model

Docs

Doc0: "I love AI" → tokens = ["i","love","ai"], bigrams = ["i love","love ai"]

Doc1: "I love love ML" → tokens = ["i","love","love","ml"], bigrams = ["i love","love love","love ml"]

Vocabulary (fixed order)

["ai", "i", "love", "ml", "i love", "love ai", "love love", "love ml"]
   0     1     2      3      4        5          6           7   (indices)

1) Count vectors (TF)

Doc0 terms used: ["i","love","ai","i love","love ai"]
→ Counts by vocab index:

ai(0)=1, i(1)=1, love(2)=1, ml(3)=0, i love(4)=1, love ai(5)=1, love love(6)=0, love ml(7)=0
Doc0 TF: [1, 1, 1, 0, 1, 1, 0, 0]

Doc1 terms used: ["i","love","love","ml","i love","love love","love ml"]
→ Counts by vocab index:

ai(0)=0, i(1)=1, love(2)=2, ml(3)=1, i love(4)=1, love ai(5)=0, love love(6)=1, love ml(7)=1
Doc1 TF: [0, 1, 2, 1, 1, 0, 1, 1]

2) Document frequency (df) and IDF (sklearn style)

Number of docs N = 2.

df: how many docs contain the term at least once

ai=1, i=2, love=2, ml=1, i love=2, love ai=1, love love=1, love ml=1
df: [1,2,2,1, 2,1,1,1]

sklearn idf (with smoothing):

idf(t)=ln⁡(1+N1+df(t))+1
idf(t)=ln(
1+df(t)
1+N
	​

)+1

If df=1 → idf = ln(3/2)+1 ≈ 1.4055

If df=2 → idf = ln(3/3)+1 = 1.0

IDF per term:
[1.4055, 1.0, 1.0, 1.4055, 1.0, 1.4055, 1.4055, 1.4055]

3) TF-IDF (no sublinear TF) and L2 normalization
Doc0 TF-IDF (before norm)

Multiply TF × IDF termwise:

ai: 1 × 1.4055 = 1.4055

i: 1 × 1.0 = 1.0

love: 1 × 1.0 = 1.0

ml: 0 × 1.4055 = 0.0

i love: 1 × 1.0 = 1.0

love ai: 1 × 1.4055 = 1.4055

love love: 0 × 1.4055 = 0.0

love ml: 0 × 1.4055 = 0.0

Doc0 TF-IDF: [1.4055, 1.0, 1.0, 0.0, 1.0, 1.4055, 0.0, 0.0]
L2 norm = √(∑ squares) ≈ 2.6364
Doc0 (L2-normalized):
[0.533, 0.379, 0.379, 0.000, 0.379, 0.533, 0.000, 0.000]

Doc1 TF-IDF (before norm)

ai: 0 × 1.4055 = 0.0

i: 1 × 1.0 = 1.0

love: 2 × 1.0 = 2.0

ml: 1 × 1.4055 = 1.4055

i love: 1 × 1.0 = 1.0

love ai: 0 × 1.4055 = 0.0

love love: 1 × 1.4055 = 1.4055

love ml: 1 × 1.4055 = 1.4055

Doc1 TF-IDF: [0.0, 1.0, 2.0, 1.4055, 1.0, 0.0, 1.4055, 1.4055]
L2 norm ≈ 3.4534
Doc1 (L2-normalized):
[0.000, 0.290, 0.579, 0.407, 0.290, 0.000, 0.407, 0.407]

Final matrix X (rows=docs, cols=vocab)
X = [
  [0.533, 0.379, 0.379, 0.000, 0.379, 0.533, 0.000, 0.000],   # Doc0
  [0.000, 0.290, 0.579, 0.407, 0.290, 0.000, 0.407, 0.407]    # Doc1
]
y = [1, 0]



In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Vocab order for reference:
# ["ai", "i", "love", "ml", "i love", "love ai", "love love", "love ml"]

X = np.array([
    [0.533, 0.379, 0.379, 0.000, 0.379, 0.533, 0.000, 0.000],  # Doc0
    [0.000, 0.290, 0.579, 0.407, 0.290, 0.000, 0.407, 0.407],  # Doc1
])
y = np.array([1, 0])

clf = LogisticRegression(max_iter=1000)
clf.fit(X, y)
pred = clf.predict(X)
proba = clf.predict_proba(X)[:, 1]

print("Pred:", pred.tolist())
print("Proba positive:", np.round(proba, 3).tolist())
print("Train acc:", accuracy_score(y, pred))
print("Weights:", np.round(clf.coef_[0], 3).tolist())
print("Bias:", round(clf.intercept_[0], 3))


Pred: [1, 0]
Proba positive: [0.561, 0.439]
Train acc: 1.0
Weights: [0.234, 0.039, -0.088, -0.179, 0.039, 0.234, -0.179, -0.179]
Bias: 0.0


TF-IDF (uni + bi-grams) → Logistic Regression (what happened)
	1.	Build vocabulary
Include unigrams (“ai”, “i”, “love”, “ml”) and bigrams (“i love”, “love ai”, “love love”, “love ml”). Fix a column order.
	2.	Count vectors (TF)
For each doc, count how many times each vocab term appears.
	•	Doc0 has counts for “ai”, “i”, “love”, “i love”, “love ai”.
	•	Doc1 has counts for “i”, “love” (twice), “ml”, “i love”, “love love”, “love ml”.
	3.	Document frequency (df)
For each term, how many documents contain it (presence/absence, not how often).
	4.	Compute IDF (smoothed)
Use sklearn’s idf = ln((1+N)/(1+df)) + 1.
	•	Common terms across both docs get idf ≈ 1.
	•	Terms appearing in only one doc get a higher idf (>1).
	5.	TF-IDF weighting
Multiply each term’s count by its idf. This downweights ubiquitous terms and upweights rarer, more discriminative ones.
	6.	L2 normalize each document vector
Scale each document’s TF-IDF vector to unit length so comparisons aren’t dominated by longer texts.
	7.	Train Logistic Regression
Feed the 2×(vocab_size) TF-IDF matrix to Logistic Regression.
	•	It learns a weight per term (unigram/bigram).
	•	Positive weights: features pushing toward label 1; negative toward label 0.

Key idea: TF-IDF is still sparse and interpretable—each dimension is a word (or phrase). N-grams add short-range order (e.g., “love love” differs from two independent “love”s).